# TESTING TAPAS ATTACKS

In [15]:
!pip install -r ./requirements.txt

  Cloning https://github.com/alan-turing-institute/privacy-sdg-toolbox (to revision a7069d7e040828db0da174d1b003fa03a98e5453) to /private/var/folders/6q/2r6m8pvx5wsd73wrfykl83sh0000gn/T/pip-install-o_1w5qoc/tapas_615bb1d1a02d4b40b6558b0cf902345c
  Running command git clone --filter=blob:none --quiet https://github.com/alan-turing-institute/privacy-sdg-toolbox /private/var/folders/6q/2r6m8pvx5wsd73wrfykl83sh0000gn/T/pip-install-o_1w5qoc/tapas_615bb1d1a02d4b40b6558b0cf902345c
  Running command git rev-parse -q --verify 'sha^a7069d7e040828db0da174d1b003fa03a98e5453'
  Running command git fetch -q https://github.com/alan-turing-institute/privacy-sdg-toolbox a7069d7e040828db0da174d1b003fa03a98e5453
  Resolved https://github.com/alan-turing-institute/privacy-sdg-toolbox to commit a7069d7e040828db0da174d1b003fa03a98e5453
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached aiofiles-22.1.0-py3-none

## Common part for attacks

In [18]:
## TAPAS TRIAL INTERFACES
import tapas.datasets
import tapas.generators
import tapas.threat_models
import tapas.attacks
import tapas.report

In [19]:
# Load the data.
data = tapas.datasets.TabularDataset.read(
    "data/2011 Census Microdata Teaching File", label="Census"
)

# Selects the target record
target_record = data.get_records([1])
data.drop_records([1], in_place=True)

# Create a dummy generator (It just samples data from original dataset)
generator = tapas.generators.Raw()

# Select the auxiliary data + black-box attack model.
data_knowledge = tapas.threat_models.AuxiliaryDataKnowledge(
    data, auxiliary_split=0.5, num_training_records=1000,
)

sdg_knowledge = tapas.threat_models.BlackBoxKnowledge(
    generator, num_synthetic_records=1000,
)

## MIA

In [20]:
# Defining a threat model 
threat_model = tapas.threat_models.TargetedMIA(
    attacker_knowledge_data=data_knowledge,
    target_record=target_record,
    attacker_knowledge_generator=sdg_knowledge,
    generate_pairs=True,
    replace_target=True,
)

attack = tapas.attacks.GroundhogAttack(
    use_hist=False, use_corr=False, label="NaiveGroundhog"
)

attack.train(threat_model,num_samples=100)
summary_mia = threat_model.test(attack, num_samples=100)
report_mia = tapas.report.MIAttackReport([summary_mia])

In [21]:
report_mia.attacks_data

,dataset,target_id,generator,attack,accuracy,true_positive_rate,false_positive_rate,mia_advantage,privacy_gain,auc,effective_epsilon
0,Census (AUX),1,Raw,NaiveGroundhog,0.53,0.5,0.44,0.06,0.94,0.5472,0.916291


## AIA

In [22]:
threat_model = tapas.threat_models.TargetedAIA(
    attacker_knowledge_data=data_knowledge,
    # Specific to AIA: the sensitive attribute and its possible values.
    sensitive_attribute="Country of Birth",
    attribute_values=["-9", "1", "2"],
    target_record=target_record,
    attacker_knowledge_generator=sdg_knowledge,
)


attack =tapas.attacks.GroundhogAttack()
attack.train(threat_model, num_samples=100)

summary_aia = threat_model.test(attack, num_samples=100)
report_aia = tapas.report.MIAttackReport([summary_aia])  

/Users/olesia/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/olesia/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/olesia/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/olesia/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/olesia/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/olesia/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: inv

In [23]:
report_aia.attacks_data

,dataset,target_id,generator,attack,sensitive_attribute,accuracy
0,Census (AUX),1,Raw,Groundhog,Country of Birth,0.65


## Extending repirts with new metrics

In [24]:
from sklearn.metrics import precision_score, recall_score, f1_score

### Approach 1

In [25]:
report_mia.attacks_data["f1_score"] = f1_score(summary_mia.labels, summary_mia.predictions)
report_mia.metrics.append("f1_score") # register the metric so it's included in `.metrics`

In [26]:
print(precision_score(summary_aia.labels, summary_aia.predictions, average="macro"), recall_score(summary_aia.labels, summary_aia.predictions, average="weighted"), f1_score(summary_aia.labels, summary_aia.predictions, average="micro"))
report_aia.attacks_data["precision"] =  precision_score(summary_aia.labels, summary_aia.predictions, average="macro")
report_aia.attacks_data["recall"] =  recall_score(summary_aia.labels, summary_aia.predictions, average="weighted")
report_aia.attacks_data["f1_score"] =  f1_score(summary_aia.labels, summary_aia.predictions, average="micro")

report_aia.metrics.append("precision") # register the metric so it's included in `.metrics`
report_aia.metrics.append("recall") # register the metric so it's included in `.metrics`
report_aia.metrics.append("f1_score") # register the metric so it's included in `.metrics`
report_aia.attacks_data

0.594079457364341 0.65 0.65


,dataset,target_id,generator,attack,sensitive_attribute,accuracy,precision,recall,f1_score
0,Census (AUX),1,Raw,Groundhog,Country of Birth,0.65,0.594079,0.65,0.65


### Approach 2

In [27]:
import tapas
import pandas as pd

In [28]:
# extra metrics 
def f1_micro(y_true, y_pred):
    return f1_score(y_true, y_pred, average="micro")

def recall_weighted(y_true, y_pred):
    return recall_score(y_true, y_pred, average="weighted")

#### MIA

In [29]:
class ExtendedAttackSummary():
    
    def __init__(self, extra_metrics):
        self.extra_metrics = extra_metrics
    
    def get_metrics(self):
        """
        Calculates all MIA relevant metrics and returns it as a dataframe.

        Returns
        -------
        A dataframe
            A dataframe with attack info and metrics.  The dataframe has the following structure.
            Index:
                RangeIndex
            Columns:
                accuracy: float
                ... metrics passed as extra_metrics argument

        """
        if len(self.extra_metrics) == 0:
            return pd.DataFrame()
         
        return pd.DataFrame([[ex(self.labels, self.predictions) for ex in self.extra_metrics]], columns=[ex.__name__ for ex in self.extra_metrics],)

In [30]:
class MIAttackSummary(tapas.report.attack_summary.MIAttackSummary, ExtendedAttackSummary):
    """
    Class summarising main performance metrics of a label-inference attack with additionally passed metrics

    """

    def __init__(self,
                 labels,
                 predictions,
                 scores=None,
                 generator_info="",
                 attack_info="",
                 dataset_info="",
                 target_id="",
                 extra_metrics=[], ):
        """
        Parameters
        ----------
        labels: list[int]
            List with true labels of the target membership in the dataset.
        predictions: list[int]
            List with the predicted labels of the target membership in the dataset.
        scores: list[float]
            List with the scores related to each prediction.
        extra_metrics: list[function or Metric]
            list of function for metrics calculation 
        """
        tapas.report.attack_summary.MIAttackSummary.__init__(self, labels, predictions, scores, generator_info, attack_info, dataset_info, target_id)
        ExtendedAttackSummary.__init__(self, extra_metrics)
        

    def get_metrics(self):
        """
        Calculates all MIA relevant metrics and returns it as a dataframe.

        Returns
        -------
        A dataframe
            A dataframe with attack info and metrics.  The dataframe has the following structure.
            Index:
                RangeIndex
            Columns:
                accuracy: float
                ... metrics passed as extra_metrics argument

        """
        return pd.concat(
                [tapas.report.attack_summary.MIAttackSummary.get_metrics(self), 
                 ExtendedAttackSummary.get_metrics(self)], axis=1
        )


In [31]:
class TargetedMIA(tapas.threat_models.TargetedMIA):

    def __init__(self, extra_metrics =[], **kwargs ):
        super().__init__(**kwargs)
        self.extra_metrics = extra_metrics
        
    def _wrap_output(self, truth_labels, pred_labels, scores, attack):
        if len(self.extra_metrics) == 0:
            return super()._wrap_output(truth_labels, pred_labels, scores, attack)
        
        return MIAttackSummary(
            truth_labels,
            pred_labels,
            scores,
            generator_info=self.atk_know_gen.label,
            attack_info=attack.label,
            dataset_info=self.atk_know_data.label,
            target_id=self.target_record.label,
            extra_metrics = self.extra_metrics
        )

In [32]:
# Defining a threat model 
threat_model_mia_ext = TargetedMIA(
    attacker_knowledge_data=data_knowledge,
    target_record=target_record,
    attacker_knowledge_generator=sdg_knowledge,
    generate_pairs=True,
    replace_target=True,
    extra_metrics=[f1_micro, recall_weighted]
)
attack_mia_ext = tapas.attacks.GroundhogAttack(
    use_hist=False, use_corr=False, label="NaiveGroundhog"
)

attack_mia_ext.train(threat_model_mia_ext,num_samples=100)
summary_mia_ext = threat_model_mia_ext.test(attack_mia_ext, num_samples=100)
report_mia_ext = tapas.report.MIAttackReport([summary_mia_ext])

In [33]:
report_mia_ext.attacks_data

,dataset,target_id,generator,attack,accuracy,true_positive_rate,false_positive_rate,mia_advantage,privacy_gain,auc,effective_epsilon,f1_micro,recall_weighted
0,Census (AUX),1,Raw,NaiveGroundhog,0.54,0.34,0.26,0.08,0.92,0.5946,0.619039,0.54,0.54


#### AIA

In [34]:
class AIAttackSummary(tapas.report.attack_summary.AIAttackSummary, ExtendedAttackSummary):
    """
    Class summarising main performance metrics of a label-inference attack with additionally passed metrics

    """

    def __init__(self, 
                 labels, 
                 predictions, 
                 scores, 
                 generator_info="",
                 attack_info="",
                 dataset_info="",
                 target_id="",
                 sensitive_attribute="",
                 extra_metrics=[], ):
        """
        Parameters
        ----------
        labels: list[int]
            List with true labels of the target membership in the dataset.
        predictions: list[int]
            List with the predicted labels of the target membership in the dataset.
        scores: list[float]
            List with the scores related to each prediction.
        extra_metrics: list[function or Metric]
            list of function for metrics calculation 
        """
        tapas.report.attack_summary.AIAttackSummary.__init__(self, labels, predictions, scores, generator_info, attack_info, dataset_info, target_id, sensitive_attribute, )
        ExtendedAttackSummary.__init__(self, extra_metrics)
        

    def get_metrics(self):
        """
        Calculates all MIA relevant metrics and returns it as a dataframe.

        Returns
        -------
        A dataframe
            A dataframe with attack info and metrics.  The dataframe has the following structure.
            Index:
                RangeIndex
            Columns:
                accuracy: float
                ... metrics passed as extra_metrics argument

        """
        return pd.concat(
                [tapas.report.attack_summary.AIAttackSummary.get_metrics(self), 
                 ExtendedAttackSummary.get_metrics(self)], axis=1
        )


In [35]:
class TargetedAIA(tapas.threat_models.TargetedAIA):

    def __init__(self, extra_metrics =[], **kwargs ):
        super().__init__(**kwargs)
        self.extra_metrics = extra_metrics
        
    def _wrap_output(self, truth_labels, pred_labels, scores, attack):
        if len(self.extra_metrics) == 0:
            return super()._wrap_output(truth_labels, pred_labels, scores, attack)
        
        return AIAttackSummary(
            truth_labels,
            pred_labels,
            scores,
            generator_info=self.atk_know_gen.label,
            attack_info=attack.label,
            dataset_info=self.atk_know_data.label,
            target_id=self.target_record.label,
            sensitive_attribute=self.sensitive_attribute,
            extra_metrics = self.extra_metrics
        )

In [36]:
threat_model_extended = TargetedAIA(
    attacker_knowledge_data=data_knowledge,
    # Specific to AIA: the sensitive attribute and its possible values.
    sensitive_attribute="Country of Birth",
    attribute_values=["-9", "1", "2"],
    target_record=target_record,
    attacker_knowledge_generator=sdg_knowledge,
    extra_metrics=[f1_micro, recall_weighted]
)

attack_extended =tapas.attacks.GroundhogAttack()
attack_extended.train(threat_model_extended, num_samples=100)

summary_extended = threat_model_extended.test(attack_extended, num_samples=100)
report_extended = tapas.report.MIAttackReport([summary_extended])  
report_extended.attacks_data

/Users/olesia/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/olesia/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/olesia/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/olesia/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/olesia/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/olesia/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: inv

,dataset,target_id,generator,attack,sensitive_attribute,accuracy,f1_micro,recall_weighted
0,Census (AUX),1,Raw,Groundhog,Country of Birth,0.64,0.64,0.64


### Approach 3

In [37]:
class ExtendedAttackSummary():

    def __init__(self, original_attack_summary,  *args, **kwargs):
        self.extra_metrics = kwargs.pop("extra_metrics", None)
        self._original_instance = original_attack_summary(*args, **kwargs)
    
    def get_metrics(self):
        return pd.concat(
                [self._original_instance.get_metrics(), 
                 pd.DataFrame([[ex(self.labels, self.predictions) for ex in self.extra_metrics]], columns=[ex.__name__ for ex in self.extra_metrics],)], axis=1
        )

    def __getattr__(self, name):
        # Delegate attribute access to the original instance
        return getattr(self._original_instance, name)


def extend_threat_model(threat_model, extra_metrics):
    
    cls = type(threat_model)

    ReportClass = tapas.report.attack_summary.MIAttackSummary
    kwargs = {}
    if isinstance(threat_model, tapas.threat_models.TargetedAIA):
        kwargs = {"sensitive_attribute": threat_model.sensitive_attribute}
        if len(threat_model.attribute_values) == 2:
            ReportClass = tapas.report.attack_summary.BinaryAIAttackSummary
            kwargs["positive_value"] = threat_model.attribute_values[1]
        else:
            ReportClass = tapas.report.attack_summary.AIAttackSummary

    class ExtendedClass(cls):
        def __init__(self, super_obj, extra_metrics):
            self.extra_metrics = extra_metrics
            for k, v in vars(super_obj).items():
                setattr(self, k, v)

        def _wrap_output(self, truth_labels, pred_labels, scores, attack):
            if len(self.extra_metrics) == 0:
                return cls._wrap_output(self, truth_labels, pred_labels, scores, attack)
            
            return ExtendedAttackSummary(
                ReportClass,
                truth_labels,
                pred_labels,
                scores,
                generator_info=self.atk_know_gen.label,
                attack_info=attack.label,
                dataset_info=self.atk_know_data.label,
                target_id=self.target_record.label,
                extra_metrics = self.extra_metrics,
                **kwargs
            )
        
    return ExtendedClass(threat_model, extra_metrics)
    




In [38]:
threat_model_extended = tapas.threat_models.TargetedAIA(
    attacker_knowledge_data=data_knowledge,
    # Specific to AIA: the sensitive attribute and its possible values.
    sensitive_attribute="Country of Birth",
    attribute_values=["-9", "1", "2"],
    target_record=target_record,
    attacker_knowledge_generator=sdg_knowledge
)
threat_model_extended = extend_threat_model(threat_model_extended, [f1_micro, recall_weighted])

attack_extended =tapas.attacks.GroundhogAttack()
attack_extended.train(threat_model_extended, num_samples=100)

summary_extended = threat_model_extended.test(attack_extended, num_samples=100)
report_extended = tapas.report.MIAttackReport([summary_extended])  
report_extended.attacks_data

/Users/olesia/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/olesia/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/olesia/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/olesia/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/olesia/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/olesia/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: inv

,dataset,target_id,generator,attack,sensitive_attribute,accuracy,f1_micro,recall_weighted
0,Census (AUX),1,Raw,Groundhog,Country of Birth,0.68,0.68,0.68


In [39]:
# Defining a threat model 
threat_model_mia_ext = tapas.threat_models.TargetedMIA(
    attacker_knowledge_data=data_knowledge,
    target_record=target_record,
    attacker_knowledge_generator=sdg_knowledge,
    generate_pairs=True,
    replace_target=True,
)

threat_model_mia_ext = extend_threat_model(threat_model_mia_ext, [f1_micro, recall_weighted])

attack_mia_ext = tapas.attacks.GroundhogAttack(
    use_hist=False, use_corr=False, label="NaiveGroundhog"
)

attack_mia_ext.train(threat_model_mia_ext,num_samples=100)
summary_mia_ext = threat_model_mia_ext.test(attack_mia_ext, num_samples=100)
report_mia_ext = tapas.report.MIAttackReport([summary_mia_ext])
report_mia_ext.attacks_data

,dataset,target_id,generator,attack,accuracy,true_positive_rate,false_positive_rate,mia_advantage,privacy_gain,auc,effective_epsilon,f1_micro,recall_weighted
0,Census (AUX),1,Raw,NaiveGroundhog,0.53,0.6,0.54,0.06,0.94,0.5618,0.916291,0.53,0.53
